In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
CREATE WIDGET TEXT srcData1 DEFAULT "adlsrcset1";
CREATE WIDGET TEXT srcData2 DEFAULT "adlsrcset2";

In [0]:
%python
srcData1 = dbutils.widgets.get("srcData1")
srcData2 = dbutils.widgets.get("srcData2")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${srcData1}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw-fuente1`
URL 'abfss://raw@${srcData1}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para la Fuente 1 (Clientes en adlsrcset1)';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw-fuente2`
URL 'abfss://raw@${srcData2}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para la Fuente 2 (Transacciones en adlsrcset2)';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${srcData1}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${srcData1}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${srcData1}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${srcData1}.dfs.core.windows.net/'
COMMENT 'Catálogo Medallion para Fraude Bancario en Perú';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{srcData1}.dfs.core.windows.net/", True)
dbutils.fs.rm(f"abfss://silver@{srcData1}.dfs.core.windows.net/", True)
dbutils.fs.rm(f"abfss://golden@{srcData1}.dfs.core.windows.net/", True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.clientes (
    cliente_id long,
    edad integer,
    ingreso_mensual double,
    antiguedad_bancaria integer,
    departamento string,
    banco_principal string,
    app_preferida string,
    tipo_operacion_frecuente string,
    monto_promedio_transaccion double,
    tiene_token_fisico integer,
    tiene_token_digital integer,
    score_riesgo double,
    fecha_registro string,
    _fecha_ingesta timestamp
)
USING DELTA
LOCATION "abfss://bronze@${srcData1}.dfs.core.windows.net/clientes";

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.transacciones (
    transaccion_id string,
    cliente_id long,
    monto_operacion double,
    frecuencia_transacciones integer,
    uso_cajeros integer,
    uso_agencias integer,
    uso_pos integer,
    cambios_clave_recientes integer,
    historial_suspicious integer,
    ubicacion_inusual integer,
    horario_inusual integer,
    dispositivo_nuevo integer,
    fallas_autenticacion integer,
    intentos_contacto_sospechoso integer,
    operaciones_canceladas integer,
    alerta_sistema integer,
    fraude_confirmado integer,
    _fecha_ingesta timestamp
)
USING DELTA
LOCATION "abfss://bronze@${srcData1}.dfs.core.windows.net/transacciones";


###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.clientes_cleaned (
    cliente_id long,
    edad integer,
    ingreso_mensual double,
    antiguedad_bancaria integer,
    departamento string,
    banco_principal string,
    app_preferida string,
    tipo_operacion_frecuente string,
    monto_promedio_transaccion double,
    score_riesgo double,
    fecha_registro timestamp,
    _fecha_procesamiento timestamp
)
USING DELTA
LOCATION "abfss://silver@${srcData1}.dfs.core.windows.net/clientes_cleaned";


In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.transacciones_cleaned (
    transaccion_id string,
    cliente_id long,
    monto_operacion double,
    ubicacion_inusual integer,
    horario_inusual integer,
    dispositivo_nuevo integer,
    fallas_autenticacion integer,
    fraude_confirmado integer,
    _fecha_procesamiento timestamp
)
USING DELTA
LOCATION "abfss://silver@${srcData1}.dfs.core.windows.net/transacciones_cleaned";

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_analisis_fraude (
    transaccion_id string,
    cliente_id long,
    monto_operacion double,
    monto_promedio_transaccion double,
    ingreso_mensual double,
    departamento string,
    banco_principal string,
    app_preferida string,
    score_riesgo double,
    ubicacion_inusual integer,
    dispositivo_nuevo integer,
    fallas_autenticacion integer,
    fraude_confirmado integer,
    alerta_monto_anomalo boolean,
    nivel_riesgo_transaccion string,
    _fecha_gold timestamp
)
USING DELTA
PARTITIONED BY (departamento)
LOCATION "abfss://golden@adlsrcset1.dfs.core.windows.net/golden_analisis_fraude";